# Module 02: Airflow

## What is Airflow and why does it exist?

You have a pipeline: scrape → embed → cluster → upload → reload. Right now you run these steps manually, in order, on your laptop. This breaks in several ways:
- You forget a step or run them out of order
- One step fails and you don't know until you check the next output
- You need to run it on a schedule (every few days) while your machine may be sleeping
- You want to retry failed steps, not restart from scratch

**Airflow solves this.** It's a platform for programmatically authoring, scheduling, and monitoring workflows as **DAGs** (Directed Acyclic Graphs).

## Key Vocabulary

| Term | Meaning |
|---|---|
| **DAG** | Directed Acyclic Graph — a workflow with tasks and dependencies. "Directed" = tasks have order. "Acyclic" = no cycles (can't loop back). |
| **Task** | One unit of work in a DAG. Could be a Python function, a bash command, a Kubernetes pod. |
| **Operator** | A template for a Task type. `PythonOperator`, `BashOperator`, `KubernetesJobOperator`. |
| **DAG Run** | One execution of a DAG (can be scheduled or manually triggered). |
| **Task Instance** | One execution of one Task within one DAG Run. |
| **Scheduler** | The Airflow component that decides when to run DAGs. |
| **Executor** | How tasks actually run: `LocalExecutor` (in the same process), `KubernetesExecutor` (as K8s pods). |
| **XCom** | "Cross-communication" — a mechanism for tasks to pass small values to each other. |
| **Sensor** | A task that waits for an external condition (e.g., "wait for this S3 file to exist"). |

## The DAG Lifecycle

```
You write a DAG file (Python)
    ↓
Airflow Scheduler reads it (from dags/ folder)
    ↓
Trigger: cron schedule fires OR manual trigger
    ↓
DAG Run created
    ↓
Tasks queued in dependency order
    ↓
Executor runs each task (LocalExecutor = same machine, KubernetesExecutor = k8s pods)
    ↓
Task Instance: queued → running → success / failed / skipped
    ↓
DAG Run: success (all tasks succeeded) or failed
```

## Setup

**Option 1: Standalone (simplest for learning)**
```bash
pip install apache-airflow
airflow standalone
# Opens at http://localhost:8080, admin/admin
```

**Option 2: In your k3d cluster (as in Galaxy project)**
```bash
helm repo add apache-airflow https://airflow.apache.org
helm install airflow apache-airflow/airflow --namespace airflow --create-namespace
kubectl port-forward svc/airflow-webserver 8080:8080 -n airflow
```

For this notebook we run DAGs in standalone mode. The code is identical for the k3d setup.


---
## How to Use This Notebook

Airflow DAGs are Python files placed in the `dags/` folder that Airflow monitors. The cells below show DAG code — you should:

1. Start Airflow standalone in a terminal: `airflow standalone`
2. Find your `AIRFLOW_HOME` directory: `airflow config get-value core dags_folder`
3. Copy the DAG files from the exercises below into that `dags/` folder
4. Wait ~30 seconds for the scheduler to pick them up
5. Trigger the DAG in the Airflow UI

Each exercise includes a `%%writefile` cell to save the DAG file automatically.

In [ ]:
import os
import subprocess

# Find the Airflow dags folder
result = subprocess.run(["airflow", "config", "get-value", "core", "dags_folder"],
                        capture_output=True, text=True)
DAGS_FOLDER = result.stdout.strip()
print(f"DAGs folder: {DAGS_FOLDER}")
os.makedirs(DAGS_FOLDER, exist_ok=True)

---
## Exercise 1: Your First DAG

The minimal DAG: two tasks, one depends on the other.

In [ ]:
dag_code = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

# Every DAG starts with a DAG object
with DAG(
    dag_id="exercise_01_hello_world",          # unique name in Airflow
    description="First DAG: two simple tasks",
    schedule=None,                              # None = only run manually
    start_date=datetime(2024, 1, 1),            # required even if schedule=None
    catchup=False,                              # don't run missed past intervals
    tags=["learning", "exercise-01"],
) as dag:

    def hello():
        print("Task 1: Hello from Airflow!")
        print("This function runs as a Task Instance")
        return "hello_result"  # return values go to XCom

    def world():
        print("Task 2: World! (runs after Task 1)")

    # Create task instances from the functions
    task_hello = PythonOperator(
        task_id="say_hello",
        python_callable=hello,
    )

    task_world = PythonOperator(
        task_id="say_world",
        python_callable=world,
    )

    # Define dependency: task_world runs AFTER task_hello
    # >> means "then"
    task_hello >> task_world
'''

with open(f"{DAGS_FOLDER}/exercise_01_hello_world.py", "w") as f:
    f.write(dag_code)

print(f"DAG saved to {DAGS_FOLDER}/exercise_01_hello_world.py")
print("\nIn the Airflow UI (http://localhost:8080):")
print("1. Find 'exercise_01_hello_world' in the DAG list")
print("2. Toggle it ON (the switch on the left)")
print("3. Click the ▶ button to trigger it")
print("4. Click into the DAG run to see task logs")

---
## Exercise 2: Task Dependencies and Parallel Execution

Complex pipelines have branches (tasks that can run in parallel) and joins (tasks that wait for multiple parents). This is where the DAG structure becomes powerful.

In [ ]:
dag_code = '''
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator
import time

with DAG(
    dag_id="exercise_02_parallel_tasks",
    schedule=None,
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["learning", "exercise-02"],
) as dag:

    # This is the DAG structure we're building:
    #
    #        start
    #       /     \
    #  embed_A  embed_B      <- run in PARALLEL
    #       \     /
    #        merge           <- waits for BOTH
    #          |
    #        upload

    def start_task():
        print("Starting pipeline...")

    def embed_chunk_a():
        print("Embedding creators 0-10000...")
        time.sleep(2)  # simulate work
        print("Chunk A done")

    def embed_chunk_b():
        print("Embedding creators 10001-20000...")
        time.sleep(2)  # simulate work
        print("Chunk B done")

    def merge_embeddings():
        print("Both chunks done. Merging embeddings...")

    def upload_results():
        print("Uploading merged embeddings to storage...")

    start = PythonOperator(task_id="start", python_callable=start_task)
    embed_a = PythonOperator(task_id="embed_chunk_a", python_callable=embed_chunk_a)
    embed_b = PythonOperator(task_id="embed_chunk_b", python_callable=embed_chunk_b)
    merge = PythonOperator(task_id="merge_embeddings", python_callable=merge_embeddings)
    upload = PythonOperator(task_id="upload_results", python_callable=upload_results)

    # Fan out: start leads to embed_a AND embed_b (parallel)
    start >> [embed_a, embed_b]
    # Fan in: merge waits for BOTH embed_a AND embed_b
    [embed_a, embed_b] >> merge
    # Sequential
    merge >> upload
'''

with open(f"{DAGS_FOLDER}/exercise_02_parallel_tasks.py", "w") as f:
    f.write(dag_code)

print("Saved. Trigger in the UI and observe the parallel execution in the Graph view.")

---
## Exercise 3: XCom — Passing Data Between Tasks

XCom (cross-communication) lets tasks share small values. A task's `return` value is automatically pushed to XCom. Other tasks pull it with `ti.xcom_pull(task_ids=...)` or via Jinja templates.

**Important:** XCom is for small values (file paths, counts, status strings). Never put large data (DataFrames, embeddings) in XCom — store those in S3/Parquet and pass the path.

In [ ]:
dag_code = '''
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator

with DAG(
    dag_id="exercise_03_xcom",
    schedule=None,
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["learning", "exercise-03"],
) as dag:

    def scrape_task():
        """Scraper runs, returns path to output file."""
        # In Galaxy: this would be the real S3 path after upload
        s3_path = "s3://galaxy-raw/fandom/2024-01-15/creators.json"
        new_creator_count = 247
        print(f"Scraped {new_creator_count} creators")
        # Return value is automatically pushed to XCom under key 'return_value'
        return {"s3_path": s3_path, "new_count": new_creator_count}

    def train_task(ti):  # ti = task instance, gives access to XCom
        """Training reads the scraper output path from XCom."""
        # Pull the dict returned by scrape_task
        scrape_result = ti.xcom_pull(task_ids="scrape", key="return_value")
        print(f"Training on data from: {scrape_result[\"s3_path\"]}")
        print(f"Processing {scrape_result[\"new_count\"]} new creators")

        # Simulate silhouette score
        silhouette_score = 0.47
        return {"silhouette_score": silhouette_score, "model_version": "v2024-01-15"}

    def gate_task(ti):
        """Quality gate: only upload if silhouette score is acceptable."""
        train_result = ti.xcom_pull(task_ids="train", key="return_value")
        score = train_result["silhouette_score"]
        threshold = 0.40

        if score >= threshold:
            print(f"Score {score:.3f} >= {threshold}. Gate PASSED.")
            return "upload"  # used for branching (see Exercise 4)
        else:
            raise ValueError(f"Quality gate FAILED: silhouette={score:.3f} < {threshold}")
            # Raising an exception marks this task (and all downstream) as FAILED

    def upload_task(ti):
        train_result = ti.xcom_pull(task_ids="train", key="return_value")
        print(f"Uploading model version: {train_result[\"model_version\"]}")
        print("Upload complete.")

    scrape = PythonOperator(task_id="scrape", python_callable=scrape_task)
    train = PythonOperator(task_id="train", python_callable=train_task)
    gate = PythonOperator(task_id="quality_gate", python_callable=gate_task)
    upload = PythonOperator(task_id="upload", python_callable=upload_task)

    scrape >> train >> gate >> upload
'''

with open(f"{DAGS_FOLDER}/exercise_03_xcom.py", "w") as f:
    f.write(dag_code)

print("Saved. After triggering, check the XCom tab in each task's detail view.")

---
## Exercise 4: Scheduling with Cron

Airflow schedules DAGs using cron expressions (or presets like `@daily`, `@weekly`).

Cron format: `minute hour day-of-month month day-of-week`
- `0 2 * * 3` = 02:00 on Wednesdays
- `0 0 * * 0` = midnight on Sundays  
- `@daily` = `0 0 * * *` = midnight every day

In [ ]:
dag_code = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

# Default args apply to all tasks in the DAG
default_args = {
    "retries": 2,                    # retry failed tasks up to 2 times
    "retry_delay": timedelta(minutes=5),  # wait 5 minutes between retries
    "email_on_failure": False,        # set to True + configure email for alerts
}

with DAG(
    dag_id="exercise_04_scheduled",
    description="Runs weekly on Sundays",
    schedule="0 1 * * 0",            # 01:00 UTC every Sunday
    start_date=datetime(2024, 1, 1),
    catchup=False,                   # important: don't backfill past runs
    default_args=default_args,
    tags=["learning", "exercise-04"],
) as dag:

    def check_data(**context):
        # **context provides execution metadata
        execution_date = context["execution_date"]
        run_id = context["run_id"]
        print(f"Execution date: {execution_date}")
        print(f"Run ID: {run_id}")
        print("This task has retries=2 — if it fails, Airflow retries it twice")

    def process(**context):
        ds = context["ds"]  # execution date as string YYYY-MM-DD
        print(f"Processing data for date: {ds}")

    check = PythonOperator(task_id="check_data", python_callable=check_data)
    process = PythonOperator(task_id="process", python_callable=process)

    check >> process
'''

with open(f"{DAGS_FOLDER}/exercise_04_scheduled.py", "w") as f:
    f.write(dag_code)

print("Saved. This DAG won't auto-run until Sunday 01:00 UTC.")
print("Trigger it manually in the UI to test it immediately.")

---
## Exercise 5: TaskFlow API (modern Airflow)

Airflow 2.x introduced the TaskFlow API: use `@dag` and `@task` decorators instead of the verbose `PythonOperator` pattern. XCom passing becomes implicit — just `return` and call the function.

In [ ]:
dag_code = '''
from datetime import datetime
from airflow.decorators import dag, task

@dag(
    dag_id="exercise_05_taskflow",
    schedule=None,
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["learning", "exercise-05"],
)
def galaxy_mini_pipeline():
    """A simplified Galaxy pipeline using the TaskFlow API."""

    @task
    def scrape() -> dict:
        print("Scraping Fandom wiki...")
        return {"s3_path": "s3://galaxy/raw/creators.json", "count": 20807}

    @task
    def embed(scrape_result: dict) -> dict:
        print(f"Embedding {scrape_result[\"count\"]} creators from {scrape_result[\"s3_path\"]}")
        print("(In Galaxy: this submits a Kubernetes Job that uses the RTX 3080)")
        return {"embeddings_path": "s3://galaxy/embeddings/latest.parquet", "dims": 1024}

    @task
    def cluster(embed_result: dict) -> dict:
        print(f"Clustering embeddings from {embed_result[\"embeddings_path\"]}")
        silhouette = 0.47
        print(f"Silhouette score: {silhouette}")
        return {"model_path": "s3://galaxy/models/kmeans_v1.pkl", "silhouette": silhouette}

    @task
    def upload_and_reload(cluster_result: dict):
        if cluster_result["silhouette"] < 0.40:
            raise ValueError(f"Quality gate failed: {cluster_result[\"silhouette\"]:.3f}")
        print(f"Uploading model: {cluster_result[\"model_path\"]}")
        print("Notifying Oracle Cloud to reload data...")
        print("Pipeline complete!")

    # TaskFlow: function calls define dependencies automatically
    # XCom passing is implicit via return values
    scrape_result = scrape()
    embed_result = embed(scrape_result)
    cluster_result = cluster(embed_result)
    upload_and_reload(cluster_result)

# Instantiate the DAG
galaxy_mini_pipeline()
'''

with open(f"{DAGS_FOLDER}/exercise_05_taskflow.py", "w") as f:
    f.write(dag_code)

print("Saved. This is the pattern the real Galaxy DAGs use.")
print("Trigger in the UI and compare the Graph view to Exercise 01.")

---
## Exercise 6: KubernetesPodOperator (preview)

In the Galaxy project, the training step runs as a **Kubernetes Job** (not a Python function) because it needs GPU resources and a specific container with heavy ML dependencies. The `KubernetesPodOperator` submits a pod to your cluster and waits for it to complete.

This is the pattern that separates "Airflow as orchestrator" from "Airflow as compute" — Airflow's job is to sequence and retry, not to do the ML work itself.

In [ ]:
# This DAG is a PREVIEW — it requires a running k3d cluster to actually execute
# Study the code to understand the pattern

print("""
From the real training_pipeline_dag.py in the Galaxy project:

from airflow.providers.cncf.kubernetes.operators.pod import KubernetesPodOperator
from kubernetes.client import models as k8s
import os

@dag(schedule="0 2 * * 3", ...)
def training_pipeline_dag():

    @task
    def prepare_training_config() -> dict:
        return {
            "input_path": "oci://galaxy-raw@namespace/fandom/latest/creators.json",
            "output_path": "oci://galaxy-features@namespace/embeddings/latest.parquet",
            "n_clusters": 120,
            "mlflow_tracking_uri": "http://mlflow-service.galaxy-pipeline:5000"
        }

    # This submits a Docker container to the k3s cluster
    # The container has GTE-Large and all ML deps pre-installed
    # On desktop (COMPUTE_BACKEND=gpu): requests nvidia.com/gpu: 1
    # On M1 (COMPUTE_BACKEND=cpu): no GPU request, runs on CPU

    gpu_enabled = os.getenv("COMPUTE_BACKEND", "cpu") == "gpu"

    training_job = KubernetesPodOperator(
        task_id="training_job",
        name="galaxy-training-job",
        namespace="galaxy-pipeline",
        image="your-ecr-or-registry/galaxy-training:latest",
        cmds=["python", "train.py"],
        env_vars={
            "INPUT_PATH": "{{ ti.xcom_pull(task_ids='prepare_training_config')['input_path'] }}",
            "MLFLOW_TRACKING_URI": "http://mlflow-service.galaxy-pipeline:5000",
        },
        container_resources=k8s.V1ResourceRequirements(
            requests={"memory": "8Gi", "cpu": "2"},
            limits={"memory": "12Gi", "cpu": "4",
                    **({{"nvidia.com/gpu": "1"}} if gpu_enabled else {{}})},
        ),
        is_delete_operator_pod=True,  # clean up pod after completion
        get_logs=True,               # stream pod logs to Airflow task logs
    )
""")

---
## How Airflow Maps to the Galaxy Project

```
Local k3d cluster
└── Airflow (Helm chart, running 24/7)
    ├── fandom_scrape_dag (Weekly, Sun 01:00)
    │   ├── scrape_task         → scraper container → Oracle Object Storage
    │   └── trigger_training    → if new_count > 100, trigger training_pipeline_dag
    │
    └── training_pipeline_dag (Triggered or Wed 02:00)
        ├── training_job        → KubernetesPodOperator → GPU k3d pod
        │                          (GTE-Large + KMeans + UMAP + MLflow logging)
        ├── mlflow_gate         → check silhouette_score >= 0.45
        ├── upload_artifacts    → push Parquet to Oracle Object Storage
        ├── feast_materialize   → push features offline→online (Redis)
        └── oracle_reload       → HTTP POST to spookypharaoh.com/api/admin/reload
                                   → Oracle FastAPI → upserts Weaviate embeddings
                                   → invalidates caches
```

The key design principle: **Airflow orchestrates, it doesn't compute.** The ML work happens in the KubernetesPodOperator's container. Airflow's tasks are just coordination: check XCom, call an API, trigger another DAG. This keeps Airflow lightweight.

## Summary

| Concept | Code |
|---|---|
| Define a DAG | `@dag(dag_id=..., schedule=..., start_date=...)` |
| Define a task | `@task` or `PythonOperator(task_id=..., python_callable=...)` |
| Set dependency | `task_a >> task_b` (a then b) |
| Parallel then join | `task_a >> [task_b, task_c] >> task_d` |
| Pass data | Return from `@task`, receive as argument in next `@task` |
| Manual pull | `ti.xcom_pull(task_ids="task_name")` |
| Schedule | `schedule="0 2 * * 3"` (cron) or `"@weekly"` |